# Reprocess Any GEO Sample with Singlet

This notebook demonstrates how to reprocess an arbitrary GSM accession from GEO using the **singlet** pipeline. The workflow is:

1. **Resolve** the GSM accession → SRR run IDs via NCBI/ENA APIs
2. **Download** raw FASTQ data (ENA FTP primary, NCBI S3 fallback)
3. **Encode** FASTQs into `.1fq` archive format
4. **Process** `.1fq` → `.1pz` (align + pileup + cell calling)
5. **Load** results into AnnData for analysis

## Pipeline Output Files

The `singlet` pipeline produces compressed `.1pz` (VOCSC) sparse matrices:

| File | Type | Description |
|------|------|-------------|
| `exon_counts.1pz` | uint16 | UMI-deduplicated exon counts (genes × cells) |
| `intron_counts.1pz` | uint16 | UMI-deduplicated intron counts (genes × cells) |
| `sj_counts.1pz` | uint32 | Splice junction read counts (junctions × cells) |
| `snp_ad.1pz` | uint32 | SNP allele depth (alt-allele reads per site × cell) |
| `snp_dp.1pz` | uint32 | SNP total depth (all reads per site × cell) |
| `mt_heteroplasmy.1pz` | float | Mitochondrial variant allele frequency per cell |
| `summary.json` | — | Pipeline statistics (mapping rate, cells called, etc.) |

Optional outputs (flag-dependent):
- `possorted_genome_bam.bam` — Cell Ranger-compatible tagged BAM (`--tagged-bam`)
- `raw_feature_bc_matrix/` — unfiltered barcode matrix (`--raw-matrix`)
- `cascade_stats.json` — cascade aligner stats (`--cascade-stats`)
- `nonhost/` — viral/microbial screening results (auto if `--nonhost-db`)

## Requirements

- **`singlet` binary** — built from the monorepo (`pip install singlet` for Python, CMake build for C++ pipeline)
- **STAR genome index** — pre-built for target species (e.g. GRCh38-2024-A for human)
- **Gene annotation** — GTF file matching the genome build
- **Barcode whitelist** — auto-resolved from detected protocol (10x v2/v3, Drop-seq, etc.)
- **`curl`** and optionally **`fasterq-dump`** (SRA Toolkit) for NCBI S3 fallback

In [ ]:
import json
import os
import subprocess
import tempfile
import xml.etree.ElementTree as ET
from pathlib import Path
from urllib.request import urlopen

import singlet

## Configuration

Set the GSM accession you want to reprocess, along with paths to the pipeline binary and reference genome.

In [ ]:
# ── User configuration ──────────────────────────────────────────────
GSM_ID = "GSM2668215"                       # <-- change to any GSM accession

# Pipeline binary
SINGLET_BIN = os.environ.get(
    "SINGLET_BIN",
    str(Path.home() / "Singlet-AI/singlify/build/singlet"),
)

# Reference genome (STAR index + GTF)
REF_BASE = os.environ.get(
    "SINGLET_REF_BASE",
    "/mnt/projects/debruinz_project/cellarium/reference",
)
GENOME_DIR = f"{REF_BASE}/GRCh38-2024-A/star_2.7.11b"  # human default
EXONS_GTF  = f"{REF_BASE}/GRCh38-2024-A/genes/genes.gtf"

# Output
WORK_DIR = Path(tempfile.mkdtemp(prefix="singlet_reprocess_"))
THREADS = int(os.environ.get("SLURM_CPUS_PER_TASK", 8))

print(f"GSM:        {GSM_ID}")
print(f"Binary:     {SINGLET_BIN}")
print(f"Genome:     {GENOME_DIR}")
print(f"GTF:        {EXONS_GTF}")
print(f"Work dir:   {WORK_DIR}")
print(f"Threads:    {THREADS}")

## Step 1: Resolve GSM → SRR Run Accessions

Query NCBI E-utilities to resolve a GSM accession to its SRX experiment, then use the ENA API to get the SRR run IDs and FASTQ download URLs.

In [ ]:
def gsm_to_srr(gsm_id: str) -> list[dict]:
    """Resolve GSM → list of {srr, r1_url, r2_url} via NCBI + ENA APIs."""
    # Step 1: GSM → SRX via NCBI Entrez
    esearch_url = (
        f"https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi"
        f"?db=sra&term={gsm_id}[ACCN]&retmode=json"
    )
    esearch = json.loads(urlopen(esearch_url).read())
    uid_list = esearch["esearchresult"]["idlist"]
    if not uid_list:
        raise ValueError(f"No SRA records found for {gsm_id}")

    # Step 2: UID → SRX accession via efetch
    efetch_url = (
        f"https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi"
        f"?db=sra&id={','.join(uid_list)}&rettype=xml"
    )
    xml_data = urlopen(efetch_url).read()
    root = ET.fromstring(xml_data)

    srx_list = []
    for exp in root.iter("EXPERIMENT"):
        srx_list.append(exp.attrib.get("accession", ""))
    srx_list = [s for s in srx_list if s]
    if not srx_list:
        raise ValueError(f"No SRX accessions found for {gsm_id}")

    # Step 3: SRX → SRR + FASTQ URLs via ENA filereport
    runs = []
    for srx in srx_list:
        ena_url = (
            f"https://www.ebi.ac.uk/ena/portal/api/filereport"
            f"?accession={srx}&result=read_run&fields=run_accession,fastq_ftp"
            f"&format=json"
        )
        try:
            ena_data = json.loads(urlopen(ena_url).read())
        except Exception:
            continue
        for row in ena_data:
            srr = row["run_accession"]
            ftp_files = row.get("fastq_ftp", "").split(";")
            r1_url = r2_url = ""
            for f in ftp_files:
                if f.endswith("_1.fastq.gz"):
                    r1_url = f"ftp://{f}" if not f.startswith("ftp") else f
                elif f.endswith("_2.fastq.gz"):
                    r2_url = f"ftp://{f}" if not f.startswith("ftp") else f
            runs.append({"srr": srr, "r1_url": r1_url, "r2_url": r2_url})

    if not runs:
        raise ValueError(f"No SRR runs found for {gsm_id} (SRX: {srx_list})")
    return runs


runs = gsm_to_srr(GSM_ID)
print(f"Resolved {GSM_ID} → {len(runs)} run(s):")
for r in runs:
    print(f"  {r['srr']}")
    if r['r1_url']:
        print(f"    R1: {r['r1_url']}")
    if r['r2_url']:
        print(f"    R2: {r['r2_url']}")

## Step 2: Download FASTQ Files

Download strategy:
1. **Primary** — ENA direct FTP (fastest for most accessions)
2. **Fallback** — NCBI public S3 bucket (`sra-pub-run-odp.s3.amazonaws.com`) via the SDL API, then `fasterq-dump` to convert `.sra` → FASTQ

In [ ]:
def download_fastq(srr: str, r1_url: str, r2_url: str, out_dir: Path) -> tuple[Path, Path]:
    """Download FASTQ pair for one SRR. Falls back to NCBI S3 if ENA fails."""
    out_dir.mkdir(parents=True, exist_ok=True)
    r1_path = out_dir / f"{srr}_1.fastq.gz"
    r2_path = out_dir / f"{srr}_2.fastq.gz"

    # ── Strategy 1: ENA direct FTP ──
    if r1_url and r2_url:
        print(f"  Downloading from ENA...")
        try:
            subprocess.run(
                ["curl", "-sL", "--retry", "3", "-o", str(r1_path), r1_url],
                check=True, timeout=3600,
            )
            subprocess.run(
                ["curl", "-sL", "--retry", "3", "-o", str(r2_path), r2_url],
                check=True, timeout=3600,
            )
            if r1_path.stat().st_size > 0 and r2_path.stat().st_size > 0:
                print(f"  ✓ ENA download complete")
                return r1_path, r2_path
        except (subprocess.CalledProcessError, subprocess.TimeoutExpired) as e:
            print(f"  ENA download failed ({e}), trying S3 fallback...")

    # ── Strategy 2: NCBI public S3 (sra-pub-run-odp) ──
    print(f"  Downloading from NCBI S3...")
    sra_path = out_dir / f"{srr}.sra"

    # Try SDL API first to resolve the exact S3 URL
    sdl_url = (
        f"https://locate.ncbi.nlm.nih.gov/sdl/2/retrieve"
        f"?acc={srr}&filetype=sra"
    )
    s3_url = f"https://sra-pub-run-odp.s3.amazonaws.com/sra/{srr}/{srr}"  # fallback
    try:
        sdl_data = json.loads(urlopen(sdl_url).read())
        for result in sdl_data.get("result", []):
            for f in result.get("files", []):
                for loc in f.get("locations", []):
                    link = loc.get("link", "")
                    if "s3" in link and link.endswith(srr):
                        s3_url = link
                        break
    except Exception:
        pass  # use fallback URL

    subprocess.run(
        ["curl", "-sL", "--retry", "3", "-o", str(sra_path), s3_url],
        check=True, timeout=7200,
    )
    # Convert .sra → FASTQ
    subprocess.run(
        ["fasterq-dump", str(sra_path), "-O", str(out_dir),
         "--split-files", "-e", str(THREADS)],
        check=True, timeout=3600,
    )
    # fasterq-dump outputs uncompressed; gzip them
    for fq in out_dir.glob(f"{srr}_*.fastq"):
        subprocess.run(["gzip", str(fq)], check=True)

    r1_path = out_dir / f"{srr}_1.fastq.gz"
    r2_path = out_dir / f"{srr}_2.fastq.gz"
    if not r1_path.exists() or not r2_path.exists():
        raise FileNotFoundError(f"FASTQ conversion failed for {srr}")
    print(f"  ✓ S3 download + fasterq-dump complete")
    return r1_path, r2_path


# Download all runs
fastq_dir = WORK_DIR / "fastq"
downloaded = []
for run in runs:
    print(f"\nDownloading {run['srr']}...")
    r1, r2 = download_fastq(run["srr"], run["r1_url"], run["r2_url"], fastq_dir)
    downloaded.append((run["srr"], r1, r2))
    print(f"  R1: {r1} ({r1.stat().st_size / 1e6:.1f} MB)")
    print(f"  R2: {r2} ({r2.stat().st_size / 1e6:.1f} MB)")

## Step 3: Encode FASTQs → `.1fq` Archive

The singlet pipeline operates on `.1fq` files — a compact 2-bit packed FASTQ archive that stores barcode + UMI + cDNA reads efficiently. This step is fast and lossless.

In [ ]:
fq_archive = WORK_DIR / f"{GSM_ID}.1fq"

# Collect all R1/R2 files (multiple SRR runs get merged)
r1_files = [str(r1) for _, r1, _ in downloaded]
r2_files = [str(r2) for _, _, r2 in downloaded]

encode_cmd = [
    SINGLET_BIN, "encode",
    "--reads", *r2_files, *r1_files,   # R2 (cDNA) first, then R1 (barcode+UMI)
    "-o", str(fq_archive),
]
print(f"Encoding {len(downloaded)} run(s) → {fq_archive.name}")
print(f"  Command: {' '.join(encode_cmd)}")

result = subprocess.run(encode_cmd, capture_output=True, text=True, timeout=3600)
if result.returncode != 0:
    print(f"STDERR: {result.stderr}")
    raise RuntimeError(f"Encode failed (exit {result.returncode})")

print(f"✓ Encoded: {fq_archive} ({fq_archive.stat().st_size / 1e9:.2f} GB)")

## Step 4: Process `.1fq` → `.1pz`

This is the main pipeline step: STAR alignment → streaming pileup → UMI deduplication → EmptyDrops cell calling → `.1pz` output.

The pipeline auto-detects the single-cell protocol (10x Chromium v2/v3, Drop-seq, etc.) from the `.1fq` header.

In [ ]:
out_prefix = WORK_DIR / "output"

process_cmd = [
    SINGLET_BIN, str(fq_archive),
    "--genome-dir",  GENOME_DIR,
    "--exons",       EXONS_GTF,
    "--out-prefix",  str(out_prefix),
    "--threads",     str(THREADS),
    "--output-format", "1pz",
]

print(f"Processing {fq_archive.name}")
print(f"  Output: {out_prefix}")
print(f"  Command: {' '.join(process_cmd)}")
print()

proc = subprocess.run(
    process_cmd, capture_output=True, text=True, timeout=14400,  # 4 hour cap
)

# Print pipeline log (last 30 lines)
for line in proc.stderr.strip().splitlines()[-30:]:
    print(line)

if proc.returncode != 0:
    raise RuntimeError(f"Pipeline failed (exit {proc.returncode})")

print(f"\n✓ Pipeline complete")

## Step 5: Inspect Output Files

In [ ]:
print(f"Output directory: {out_prefix}")
print()
for f in sorted(out_prefix.iterdir()):
    size_mb = f.stat().st_size / 1e6
    print(f"  {f.name:40s}  {size_mb:8.2f} MB")

# Show pipeline summary
summary_path = out_prefix / "summary.json"
if summary_path.exists():
    print("\nPipeline Summary:")
    summary = json.loads(summary_path.read_text())
    for key, val in summary.items():
        print(f"  {key}: {val}")

## Step 6: Load Results into AnnData

The `.1pz` format can be loaded directly into an AnnData object with `singlet.read_1pz()`. The matrix is stored as a compressed sparse column (CSC) matrix.

In [ ]:
# Load exon counts (primary gene expression matrix)
exon_path = out_prefix / "exon_counts.1pz"
adata = singlet.read_1pz(str(exon_path))

print(f"Loaded: {adata}")
print(f"  Cells:    {adata.n_obs:,}")
print(f"  Genes:    {adata.n_vars:,}")
print(f"  Sparsity: {1 - adata.X.nnz / (adata.n_obs * adata.n_vars):.4%}")

In [ ]:
# Load intron counts as an additional layer
intron_path = out_prefix / "intron_counts.1pz"
if intron_path.exists():
    intron_adata = singlet.read_1pz(str(intron_path))
    adata.layers["intron"] = intron_adata.X
    print(f"Added intron layer: {intron_adata.X.nnz:,} non-zeros")

# Load splice junction counts
sj_path = out_prefix / "sj_counts.1pz"
if sj_path.exists():
    sj_adata = singlet.read_1pz(str(sj_path))
    print(f"Splice junctions: {sj_adata.n_vars:,} junctions × {sj_adata.n_obs:,} cells")

## Step 7: Basic QC

Compute standard quality control metrics.

In [ ]:
import numpy as np

# Per-cell metrics
adata.obs["total_counts"] = np.array(adata.X.sum(axis=1)).flatten()
adata.obs["n_genes"] = np.array((adata.X > 0).sum(axis=1)).flatten()

# Mitochondrial gene fraction
mt_genes = adata.var_names.str.startswith("MT-") | adata.var_names.str.startswith("mt-")
if mt_genes.any():
    adata.obs["pct_mito"] = (
        np.array(adata.X[:, mt_genes].sum(axis=1)).flatten()
        / adata.obs["total_counts"]
        * 100
    )

print(f"Median UMI/cell:   {adata.obs['total_counts'].median():,.0f}")
print(f"Median genes/cell: {adata.obs['n_genes'].median():,.0f}")
if "pct_mito" in adata.obs:
    print(f"Median % mito:     {adata.obs['pct_mito'].median():.2f}%")

## Alternative: Direct `singlet download` (Built-in SRA Streaming)

If you have SRR accessions, the singlet binary can stream directly from SRA into `.1fq` format without intermediate FASTQ files:

```bash
# Download SRA → .1fq (streams, no intermediate files)
singlet download SRR1234567 -o sample.1fq

# Process in one shot
singlet sample.1fq \
    --genome-dir /path/to/star_index \
    --exons /path/to/genes.gtf \
    --out-prefix ./output \
    --threads 8
```

## NCBI S3 Endpoint Commands

For direct AWS S3 access to NCBI SRA data (no ENA):

```bash
# Resolve S3 URL via NCBI SDL API
curl -sL "https://locate.ncbi.nlm.nih.gov/sdl/2/retrieve?acc=SRR1234567&filetype=sra"

# Direct S3 download (public, no credentials needed)
curl -sL -o SRR1234567.sra \
    "https://sra-pub-run-odp.s3.amazonaws.com/sra/SRR1234567/SRR1234567"

# Convert .sra → FASTQ (requires SRA Toolkit)
fasterq-dump SRR1234567.sra --split-files -e 8

# Encode FASTQ → .1fq
singlet encode --reads SRR1234567_2.fastq.gz SRR1234567_1.fastq.gz -o sample.1fq

# Full pipeline
singlet sample.1fq --genome-dir star_index --exons genes.gtf --out-prefix ./output
```

## Batch Processing via SLURM

```bash
#!/bin/bash
#SBATCH --cpus-per-task=8
#SBATCH --mem=64G
#SBATCH --time=4:00:00

singlet download ${SRR} -o ${WORK}/${SRR}.1fq
singlet ${WORK}/${SRR}.1fq \
    --genome-dir ${SINGLET_REF_BASE}/GRCh38-2024-A/star_2.7.11b \
    --exons ${SINGLET_REF_BASE}/GRCh38-2024-A/genes/genes.gtf \
    --out-prefix ${RESULTS}/${GSM_ID} \
    --threads ${SLURM_CPUS_PER_TASK}
```

## Cleanup

Remove intermediate files if desired.

In [ ]:
# Uncomment to clean up intermediate files:
# import shutil
# shutil.rmtree(fastq_dir, ignore_errors=True)  # Remove downloaded FASTQs
# fq_archive.unlink(missing_ok=True)             # Remove .1fq archive

print(f"Results in: {out_prefix}")
print(f"AnnData: {adata}")